# 03 · Normalization & (Optional) Batch Merge — Beginner‑Friendly

**Goal:**
1) Normalize each cohort separately (TCGA, IBD) using size factors + VST (PyDESeq2).
2) Save VST matrices for downstream scoring and DE.
3) (Optional) Create a merged view for visualization only. **Do not use merged matrices for hypothesis tests** when biology differs (e.g., tumor vs IBD).

👉 Tips:
- Run 01 and 02 notebooks first.
- If any file is missing, go back and fix manifests or paths.


In [ ]:

# === 🔧 SETTINGS ===
BASE = "/content/drive/MyDrive/Colorectal_Hippo_Dysbiosis"
print("Project base:", BASE)


In [ ]:

# Imports
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

# Optional: Install PyDESeq2 if not present (in Colab this is fine)
try:
    import PyDESeq2
except ImportError:
    !pip -q install PyDESeq2
from PyDESeq2.dds import DeseqDataSet
from PyDESeq2.deseq import DESeq2


In [ ]:

# Helper: run VST with PyDESeq2 given counts (genes x samples) and a simple condition vector
def vst_transform(counts_df, condition_series):
    # PyDESeq2 expects samples x genes
    cts = counts_df.T
    meta = pd.DataFrame({"condition": condition_series.loc[cts.index]})
    dds = DeseqDataSet(cts, meta, design_factors="condition")
    dds.deseq2()
    vst = dds.vst()
    # Return genes x samples again
    return vst.T


In [ ]:

# === Load cleaned data from previous notebooks ===
tcga_counts_p = f"{BASE}/data_processed/tcga_counts_clean.csv"
ibd_counts_p  = f"{BASE}/data_processed/ibd_counts_clean.csv"
ibd_meta_p    = f"{BASE}/data_processed/ibd_metadata_clean.csv"

tcga_exists = Path(tcga_counts_p).exists()
ibd_exists = Path(ibd_counts_p).exists() and Path(ibd_meta_p).exists()

print("TCGA counts present:", tcga_exists)
print("IBD counts & meta present:", ibd_exists)

# These will be missing locally; in Colab they exist after running 01 & 02
# Here we proceed without raising to let the notebook open cleanly.


In [ ]:

# The following cells will run once the files exist in your Drive
# (They are safe to rerun after you complete 01 & 02)
try:
    tcga_counts = pd.read_csv(f"{BASE}/data_processed/tcga_counts_clean.csv", index_col=0)
    ibd_counts  = pd.read_csv(f"{BASE}/data_processed/ibd_counts_clean.csv",  index_col=0)
    ibd_meta    = pd.read_csv(f"{BASE}/data_processed/ibd_metadata_clean.csv")
    print("Loaded shapes:", tcga_counts.shape, ibd_counts.shape, ibd_meta.shape)
except Exception as e:
    print("NOTE:", e)
    print("Run 01 and 02 notebooks first, then rerun this cell.")


In [ ]:

# === Minimal condition vectors ===
import re
try:
    id_cols = [c for c in ibd_meta.columns if re.search("sample|run|gsm|id", c, flags=re.I)]
    sid = id_cols[0] if id_cols else ibd_meta.columns[0]
    cand = [c for c in ibd_meta.columns if c.lower() in ["group","status","condition","phenotype","disease","diagnosis"]]
    ibd_group_col = cand[0] if cand else None

    ibd_meta = ibd_meta[ibd_meta[sid].isin(ibd_counts.columns)].copy()
    ibd_meta = ibd_meta.set_index(sid).loc[ibd_counts.columns]

    if ibd_group_col and ibd_group_col in ibd_meta.columns:
        ibd_condition = ibd_meta[ibd_group_col].astype(str)
    else:
        ibd_condition = pd.Series(["unknown"]*ibd_counts.shape[1], index=ibd_counts.columns, name="condition")
    print("IBD condition levels:", ibd_condition.unique()[:10])
except Exception as e:
    print("NOTE:", e)
    print("Will compute once IBD files are available.")


In [ ]:

# === VST for IBD ===
try:
    vst_ibd = vst_transform(ibd_counts, ibd_condition)
    print("VST IBD shape:", vst_ibd.shape)
    vst_ibd.to_csv(f"{BASE}/data_processed/ibd_vst.csv")
    print("Saved -> data_processed/ibd_vst.csv")
except Exception as e:
    print("NOTE:", e)


In [ ]:

# === VST for TCGA (neutral design) ===
try:
    tcga_condition = pd.Series(["neutral"]*tcga_counts.shape[1], index=tcga_counts.columns, name="condition")
    vst_tcga = vst_transform(tcga_counts, tcga_condition)
    print("VST TCGA shape:", vst_tcga.shape)
    vst_tcga.to_csv(f"{BASE}/data_processed/tcga_vst.csv")
    print("Saved -> data_processed/tcga_vst.csv")
except Exception as e:
    print("NOTE:", e)


In [ ]:

# === Optional merged visualization matrix (do not use for DE tests) ===
try:
    common_genes = vst_tcga.index.intersection(vst_ibd.index)
    v1 = vst_tcga.loc[common_genes]
    v2 = vst_ibd.loc[common_genes]

    v1_z = (v1 - v1.mean(axis=1).values[:,None]) / v1.std(axis=1, ddof=0).values[:,None]
    v2_z = (v2 - v2.mean(axis=1).values[:,None]) / v2.std(axis=1, ddof=0).values[:,None]

    merged = pd.concat([v1_z, v2_z], axis=1)
    merged.to_csv(f"{BASE}/data_processed/merged_zscore_visual_only.csv")
    print("Saved -> data_processed/merged_zscore_visual_only.csv")
except Exception as e:
    print("NOTE:", e)
    print("This step requires both VST matrices.")
